In [5]:
import pandas as pd
import numpy as np

In [3]:
s = pd.Series([1.0, 2.0, 3.0, 4.0, 5.0], index=['sensor_a', 'sensor_b', 'sensor_c', 'sensor_d', 'sensor_e'])

df = pd.DataFrame({
    'device' : ['device_1', 'device_2', 'device_3', 'device_4', 'device_5'],
    'temp_c' : [20.1, 21.3, 19.8, 22.5, 20.0],
    'active' : [True, True, False, True, False]
})

print(df.dtypes)

device        str
temp_c    float64
active       bool
dtype: object


In [4]:
df = pd.DataFrame({
    'age' : [25, 30, 35, 40, 45],
    'score' : [85, 90, 78, 92, 88],

}, index=['u1', 'u2', 'u3', 'u4', 'u5'])

sub1 = df.loc['u1':'u3',['score']]
sub2 = df.iloc[0:2, 1]

high_performers = df[(df['age']>20) & (df['score'] >= 90)]
print(high_performers)

    age  score
u2   30     90
u4   40     92


In [6]:
df = pd.DataFrame({
    'reading':[10.2, np.nan, 15.8, np.nan, 20.1],
    'status': ['OK', 'WARN', np.nan, 'OK', 'OK']
})
# count missing counts for each column
null_counts = df.isna().sum()
#Forward-fill time-series sensor gaps
df['reading_ffill'] = df['reading'].ffill()
# Statistical imputation with column mean
mean_val =df['reading'].mean()
df['reading_imputated'] = df['reading'].fillna(mean_val)
print(df[['reading', 'reading_imputated']])

   reading  reading_imputated
0     10.2          10.200000
1      NaN          15.366667
2     15.8          15.800000
3      NaN          15.366667
4     20.1          20.100000


In [7]:
users = pd.DataFrame({
    'uid' : [101, 102, 103],
    'name' : ['Alice', 'Bob', 'Charlie']
})

orders = pd.DataFrame({
    'order_id':[0-1, 0-2, 0-3],
    'user_id' : [101, 101, 104],
    'amount' : [250, 150, 300]
})

merge=pd.merge(users, orders, left_on='uid', right_on='user_id', how='left')
print(merge[['uid', 'name', 'order_id', 'amount']])

   uid     name  order_id  amount
0  101    Alice      -1.0   250.0
1  101    Alice      -2.0   150.0
2  102      Bob       NaN     NaN
3  103  Charlie       NaN     NaN


In [8]:
wide = pd.DataFrame({
    'Date' : ['2023-01-01', '2023-01-02', ],
    'sensor_A' : [10.2, 11.5],
    'sensor_B' : [20.1, 19.8]
})

long_df = pd.melt(wide, id_vars=['Date'], var_name='sensor', value_name = 'temp' )
pivoted = long_df.pivot_table(index='Date', columns = 'sensor', values ='temp', aggfunc = 'mean')
print(pivoted)

sensor      sensor_A  sensor_B
Date                          
2023-01-01      10.2      20.1
2023-01-02      11.5      19.8


In [10]:
sales = pd.DataFrame({
    'region' : ['North', 'North', 'South', 'South', 'North'],
    'product' : ['A', 'B', 'A', 'B', 'A'],
    'revenue' : [100, 150, 200, 250, 300]
})

summary = sales.groupby('region').agg(
    total_rev = ('revenue', 'sum'),
    avg_rev = ('revenue', 'mean'),
    tx_count = ('revenue', 'count')

)

print(summary)

        total_rev     avg_rev  tx_count
region                                 
North         550  183.333333         3
South         450  225.000000         2


In [13]:
df = pd.DataFrame({
    'dept' : ['HR', 'IT', 'Finance', 'IT', 'HR'],
    'salary' : [50000, 60000, 70000, 65000, 55000]
})
dept_avg = df.groupby('dept')['salary'].transform('mean')
df['salary_vs_dept'] = df['salary'] - dept_avg

zscore = lambda x: (x - x.mean()) / x.std()
df['salary_zscore'] = df.groupby('dept')['salary'].transform(zscore)

print(df[['dept', 'salary', 'salary_vs_dept']])

      dept  salary  salary_vs_dept
0       HR   50000         -2500.0
1       IT   60000         -2500.0
2  Finance   70000             0.0
3       IT   65000          2500.0
4       HR   55000          2500.0


In [14]:
ratings = pd.Series(['med', 'low', 'high', 'low', 'med', 'high'] * 1000)
mem_raw = ratings.memory_usage(deep=True)

cat_type = pd.CategoricalDtype(
    categories=['low', 'med', 'high'], ordered=True
)
cat_ratings = ratings.astype(cat_type)
mem_cat = cat_ratings.memory_usage(deep=True)

print(f"Memory: {mem_raw}B -> {mem_cat}B")
print(cat_ratings.min())

Memory: 314132B -> 6397B
low


In [16]:
arrays = [
    ['Store_1', 'Store_1', 'Store_2', 'Store_2'],
    [2024, 2025, 2024, 2025]
]
idx = pd.MultiIndex.from_arrays(arrays, names=('store', 'year'))
df = pd.DataFrame({
    'sales' : [100, 150, 200, 250]}, index =idx)

s1 = df.xs('Store_1', level='store')

wide_pannel = df.unstack(level='year')
print(wide_pannel)

        sales     
year     2024 2025
store             
Store_1   100  150
Store_2   200  250


In [17]:
raw = pd.DataFrame({
    'age' : [25, 30, 35, 40, 45],
    'income' : [0, 60000, 70000, 80000, 90000]
})

def standardize_income(df_in):
    return df_in.assign(z_inc=lambda d: (d.income - d.income.mean()) / d.income.std() )

processed = (
    raw
    .query('age >= 18')
    .assign(log_inc = lambda d: np.log1p(d['income']))
    .pipe(standardize_income)
    .sort_values(by = 'income', ascending=False)
)

print(processed[['age', 'income', 'log_inc']])

   age  income    log_inc
4   45   90000  11.407576
3   40   80000  11.289794
2   35   70000  11.156265
1   30   60000  11.002117
0   25       0   0.000000
